# Housing Strand — Is This Data Good Enough to Build a Benchmark On?

> A smart sensor in every home. About three months of hourly readings (~90 days).
> Fourteen households with broken sensors. Timestamps that disagree with reality.
> **Before anyone builds a model on this data, someone needs to audit it. That's your team.**

**Your task:** You are the data engineering team reviewing whether this social housing
dataset is trustworthy enough to serve as the foundation for a national energy
forecasting benchmark.

---

## How your team divides the work

| Track | Role | Who takes it | Feeds into |
|---|---|---|---|
| **Track A — The Sensor Inspector** | 1–2 people | Profile dropout, drift, coverage gaps, and the property-metadata issues (`reference` formatting, `Sub-building` completeness, `address` standardisation, `avgCo2` outages) per household | Data quality flags in the benchmark card |
| **Track B — The Split Builder** | 1 person | Design splits that prevent data leakage between households, stratified across property type, postcode area, and reference type | Recommended split strategy in the card |
| **Track C — The Equity Analyst** | 1–2 people | Find which households the forecasting model fails — and who lives in them | Equity findings and social impact assessment |

Track B can be done by one person in ~1 hour, then they join Part 2.

See [data/README.md](../data/README.md) for a full guide to the dataset.

---

## This notebook covers Phase 1 + Phase 2 (~3 hours)

**Phase 1 — all together (30 min):** Run the data exploration below. Everyone should
understand the deliberate flaws before splitting into roles.

**Phase 2 — split by role (~2.5 hours):** Track A profiles household sensor quality.
Track B builds benchmark splits. Track C explores the forecasting equity gap.

When Phase 2 is done, open `02_equity_and_benchmark.ipynb` together.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

df = pd.read_csv('../data/raw/housing_synthetic_120hh.csv',
                 parse_dates=['timestamp_recorded','timestamp_actual'])
print(f'Shape: {df.shape}')
print(f'Households: {df.household_id.nunique()}')
print(f'Date range: {df.timestamp_actual.min()} → {df.timestamp_actual.max()}')
df.head(3)

## Get to know your data (~25 min)

Before building anything, understand what you are working with.

The dataset has ~260,000 rows — 120 households × 2,160 hourly readings each. Each property is keyed by `household_id` and carries the metadata a housing council would actually hold: a `reference`, optional `Sub-building`, full `address`, and `postcode`. Hourly sensor readings (smart meter, indoor temperature, CO₂, noise, comfort survey) sit alongside daily-averaged environmental readings (`avgTemperature`, `avgHumidity`, `avgCo2`) that share the same value across the 24 hours of each day.

Sensor modalities have very different reliability. Smart meter data is almost always present. The noise sensor drops out frequently. Thermal comfort surveys are extremely sparse — most rows have none at all. `avgCo2` has significant gaps, including multi-day outages on some properties.

**Work through these questions — run the cells below and note what you see:**

1. What is the null rate for each sensor modality, including `avgCo2`?
2. How many households have timestamp drift? Can you detect it programmatically?
3. Does survey response rate vary by property type (flat vs terraced vs semi-detached)?
4. Are there households where sensors are offline for days at a time?
5. How clean is the property metadata? Is `reference` consistent? Is every property fully addressed?

See [data/README.md](../data/README.md) for column descriptions and known flaws.

In [ ]:
# == TASK 1: Null rates per modality ==
modalities = ['smart_meter_kwh','indoor_temp_c','co2_ppm','noise_db','survey_thermal_comfort']
daily_aggregates = ['avgTemperature','avgHumidity','avgCo2']

print('Overall null rates (hourly modalities):')
print(df[modalities].isna().mean().round(3))

print('\nOverall null rates (daily aggregates):')
print(df[daily_aggregates].isna().mean().round(3))

# Per-household null rates (reveals unreliable households)
hh_nulls = df.groupby('household_id')[modalities + daily_aggregates].apply(lambda x: x.isna().mean())
print('\nHouseholds with >40% indoor_temp_c dropout:')
print(hh_nulls[hh_nulls['indoor_temp_c']>0.40].head())

print('\nHouseholds with >30% avgCo2 outage:')
print(hh_nulls[hh_nulls['avgCo2']>0.30][['avgCo2']].head())

In [ ]:
# == TASK 2: Detect timestamp drift ==
# Drift = recorded timestamp differs from actual timestamp
df['timestamp_delta_mins'] = (df['timestamp_recorded'] - df['timestamp_actual']).dt.total_seconds() / 60
print('Timestamp delta stats:')
print(df['timestamp_delta_mins'].describe())
print(f'\nRows with drift > 0: {(df.timestamp_delta_mins != 0).mean():.1%}')

# Which households have it?
drift_hh = df[df.timestamp_delta_mins > 0].groupby('household_id').size()
print(f'\nHouseholds with drift detected: {len(drift_hh)} / {df.household_id.nunique()}')

In [ ]:
# == TASK 3: Survey response rate by property type ==
survey_rows = df[df.survey_thermal_comfort.notna()]
print('Survey responses per household:')
print(df.groupby('household_id').survey_thermal_comfort.count().value_counts())
print('\nSurvey response by property type:')
print(survey_rows.groupby('property_type').household_id.nunique() /
      df.groupby('property_type').household_id.nunique())

## Phase 2 — Build your track (~2.5 hours)

**Split your team here.** Track A: start on the sensor and metadata quality profiler.
Track B: start on the benchmark split generator. Track C: explore the equity gap.

---

### Track A — The Sensor Inspector: build a quality profiler

For each household, produce a structured data quality report covering:

- **Dropout rates per modality** — overall and per-household worst case (including `avgCo2`)
- **Timestamp drift detection** — which households have misaligned timestamps?
  From which hour did the drift begin?
- **Coverage gaps** — any households with continuous silent periods (>24 hours)?
- **`reference` format audit** — count structured IDs vs free-text entries; flag non-conforming values
- **`Sub-building` completeness** — how many properties are missing this field?
- **`address` standardisation** — leading/trailing whitespace, mixed case, missing city
- **`avgCo2` outage profiling** — per-household total outage days and longest continuous outage
- **Quality flags** — OK / MODERATE / HIGH_DROPOUT / DRIFTED for each household

**Output:** `reference/data_profile.json` — a structured quality report.
The starter function and follow-on cells below get you most of the way there.

---

### Track B — The Split Builder: create leakage-free benchmark splits

Implement a split strategy that ensures no household appears in both training
and test sets — the most common source of data leakage in time-series benchmarks:

- **Household-disjoint split**: 70% of households in train, 15% in val, 15% in test
- **Stratification**: ensure property type, postcode area, and reference type are
  represented proportionally in all three splits
- **Leakage check**: assert zero overlap between train and test household IDs

**Output:** three CSV index files and a split rationale document.

---

### Track C — The Equity Analyst: find who the model fails

The `housing_with_forecasts.csv` file has predictions from a household-type mean
model alongside the property metadata. Explore whether it performs equally well
across all household types:

- **Mean Bias Error (MBE) by property type** — is the model systematically
  over- or under-forecasting for flats? For large households?
- **Visualise**: scatter plot of actual vs forecast, coloured by property type
- **Flag the gap**: which group has the worst bias, and what is the magnitude?
- **Additional equity dimensions** that feed into Part 2:
  - by **postcode area** (geographic clustering)
  - by **`reference` type** (structured ID vs free-text — a proxy for property registration era)
  - **coverage equity**: does `avgCo2` missingness correlate with property type?

**Output:** a chart and a brief equity summary. This feeds directly into Part 2.

In [ ]:
# == TRACK A STARTER: sensor + metadata quality profile ==
def compute_data_profile(df, modalities):
    profile = {'modalities': {}}
    for mod in modalities:
        null_rate = df[mod].isna().mean()
        hh_max_null = df.groupby('household_id')[mod].apply(lambda x: x.isna().mean()).max()
        profile['modalities'][mod] = {
            'overall_null_rate':   round(float(null_rate), 4),
            'worst_household_null': round(float(hh_max_null), 4),
            'flag': 'HIGH_DROPOUT' if null_rate > 0.15 else ('MODERATE' if null_rate > 0.05 else 'OK')
        }
    drifted = df[df.timestamp_delta_mins != 0].groupby('household_id').first().index.tolist()
    profile['timestamp_drift'] = {'affected_households': len(drifted), 'household_ids': drifted[:5]}

    props = df.drop_duplicates('household_id')
    structured = props['reference'].apply(lambda s: bool(re.match(r'^U\d{6}$', str(s)))).sum()
    freetext   = props['reference'].apply(lambda s: bool(re.match(r'^\d+\s+[a-zA-Z]', str(s)))).sum()
    sub_blank  = (props['Sub-building'].fillna('').astype(str).str.strip() == '').sum()
    addr       = props['address'].astype(str)
    profile['property_metadata'] = {
        'reference_structured_ids':  int(structured),
        'reference_free_text':       int(freetext),
        'reference_non_conforming':  int(len(props) - structured - freetext),
        'sub_building_blank':        int(sub_blank),
        'sub_building_blank_pct':    round(float(sub_blank/len(props)), 4),
        'address_whitespace_padded': int((addr != addr.str.strip()).sum()),
        'address_uppercase':         int(addr.str.contains(r'[A-Z]{3,}').sum()),
        'address_missing_comma':     int((~addr.str.contains(',')).sum()),
    }

    daily = df.drop_duplicates(['household_id','year','month','day'])[
        ['household_id','year','month','day','avgCo2']
    ].sort_values(['household_id','year','month','day'])
    per_hh_missing_days = daily.groupby('household_id')['avgCo2'].apply(lambda s: s.isna().sum())

    def longest_run(s):
        mask = s.isna().to_numpy()
        best = run = 0
        for v in mask:
            run = run + 1 if v else 0
            best = max(best, run)
        return int(best)
    per_hh_longest = daily.groupby('household_id')['avgCo2'].apply(longest_run)
    profile['avgCo2_outages'] = {
        'overall_null_rate':       round(float(df['avgCo2'].isna().mean()), 4),
        'worst_household_days':    int(per_hh_missing_days.max()),
        'longest_continuous_days': int(per_hh_longest.max()),
        'households_with_7plus_day_outage': int((per_hh_longest >= 7).sum()),
    }
    return profile

modalities = ['smart_meter_kwh','indoor_temp_c','co2_ppm','noise_db','survey_thermal_comfort','avgCo2']
profile = compute_data_profile(df, modalities)
import json
print(json.dumps(profile, indent=2))

### Track A — Audit deep-dive (examples)

The cells below surface concrete examples of the data quality issues so you can decide which ones to flag in the benchmark card. Run them, scan the output, and copy-paste any especially bad rows into the `known_limitations` list of your card.

In [ ]:
# == TRACK A: Property metadata inspection (reference, Sub-building, address) ==
props = df.drop_duplicates('household_id')[
    ['household_id','reference','Sub-building','address','postcode']
].reset_index(drop=True)

print('--- reference: non-conforming examples ---')
non_conforming = props[
    ~props['reference'].apply(lambda s: bool(re.match(r'^U\d{6}$', str(s)))) &
    ~props['reference'].apply(lambda s: bool(re.match(r'^\d+\s+[a-zA-Z]', str(s))))
]
if len(non_conforming):
    print(non_conforming[['household_id','reference']].head(10).to_string(index=False))
else:
    print('(none — every reference matches either structured or free-text pattern)')

print('\n--- Sub-building: blank examples ---')
blank_sub = props[props['Sub-building'].fillna('').astype(str).str.strip() == '']
print(f'{len(blank_sub)} properties have a blank Sub-building. First 5:')
print(blank_sub[['household_id','reference','address']].head().to_string(index=False))

print('\n--- address: formatting issues ---')
addr = props['address'].astype(str)
issues = {
    'whitespace_padded': props[addr != addr.str.strip()][['household_id','address']],
    'uppercase_street':  props[addr.str.contains(r'[A-Z]{3,}')][['household_id','address']],
    'missing_comma':     props[~addr.str.contains(',')][['household_id','address']],
}
for label, sub_df in issues.items():
    print(f'\n  {label}: {len(sub_df)} properties; first 3:')
    print(sub_df.head(3).to_string(index=False))

In [ ]:
# == TRACK A: avgCo2 outage profiling ==
daily = (df.drop_duplicates(['household_id','year','month','day'])
           [['household_id','year','month','day','avgCo2']]
           .sort_values(['household_id','year','month','day']))

def longest_run(s):
    mask = s.isna().to_numpy()
    best = run = 0
    for v in mask:
        run = run + 1 if v else 0
        best = max(best, run)
    return int(best)

outage = daily.groupby('household_id')['avgCo2'].agg(
    total_missing_days=lambda s: int(s.isna().sum()),
    longest_continuous_days=longest_run,
    total_days=lambda s: int(len(s)),
).reset_index()
outage['missing_pct'] = (outage['total_missing_days'] / outage['total_days']).round(3)

print('Per-household avgCo2 outage summary (worst 10):')
print(outage.sort_values('longest_continuous_days', ascending=False).head(10).to_string(index=False))

print(f'\nHouseholds with >= 7-day continuous avgCo2 outage: '
      f'{(outage.longest_continuous_days >= 7).sum()} / {len(outage)}')
print(f'Max single-property outage length: {outage.longest_continuous_days.max()} days')

In [ ]:
# == TRACK B STARTER: Household-disjoint split ==
def make_household_disjoint_split(df, train_frac=0.70, val_frac=0.15, seed=42):
    rng = np.random.default_rng(seed)
    households = np.asarray(df['household_id'].unique())
    rng.shuffle(households)
    n = len(households)
    n_train = int(n * train_frac)
    n_val   = int(n * val_frac)
    train_hh = households[:n_train]
    val_hh   = households[n_train:n_train+n_val]
    test_hh  = households[n_train+n_val:]
    assert len(set(train_hh) & set(test_hh)) == 0, 'LEAKAGE: households in both train and test'
    return {
        'train': df[df.household_id.isin(train_hh)].index.tolist(),
        'val':   df[df.household_id.isin(val_hh)].index.tolist(),
        'test':  df[df.household_id.isin(test_hh)].index.tolist(),
        'n_train_hh': len(train_hh), 'n_val_hh': len(val_hh), 'n_test_hh': len(test_hh),
        'train_hh': list(train_hh), 'val_hh': list(val_hh), 'test_hh': list(test_hh),
    }

split = make_household_disjoint_split(df)
print(f"Train: {split['n_train_hh']} households | Val: {split['n_val_hh']} | Test: {split['n_test_hh']}")
print('Zero-leak check passed:', len(set(split['train']) & set(split['test'])) == 0)

# Stratification check — make sure each equity dimension is represented per fold
props = df.drop_duplicates('household_id').set_index('household_id')
props['ref_type'] = props['reference'].apply(
    lambda s: 'structured' if re.match(r'^U\d{6}$', str(s)) else 'free_text'
)
props['postcode_area'] = props['postcode'].astype(str).str.extract(r'^([A-Z]+)', expand=False)
for fold_name in ['train_hh','val_hh','test_hh']:
    fold = props.loc[split[fold_name]]
    print(f"\n  {fold_name}: property_type={dict(fold.property_type.value_counts())}, "
          f"ref_type={dict(fold.ref_type.value_counts())}, "
          f"postcode_areas={fold.postcode_area.nunique()}")

## Prepare your 5-minute demo (~20 min)

Fill in [../OMAIB_PATHWAY_PAGE.md](../OMAIB_PATHWAY_PAGE.md) before judging.

Then make sure you can answer these three questions out loud:

1. **Show the worst household.** Which one has the most data quality problems, and what are they?
2. **Show the equity gap.** Which property type does the model under-forecast, and by how much?
3. **Is this dataset fit for purpose?** Would you approve it as a public benchmark?

Judges score on: **evidence quality** (are findings quantified, not just described?),
**social clarity** (could a housing officer understand your equity statement?), and
**deployability** (could the Brunel team publish this card alongside a dataset release?).

Save your output files to the `reference/` folder before the session ends.